<div dir="rtl" align="right">

# تهيئةُ مجموعةِ البياناتِ لِتعلّمِ الآلةِ

**مجموعةُ البياناتِ**: MOABB BNCI2014-001 (تخيّلٌ حركيّ)  
**القنواتُ**: 22 قناةً  
**معدّلُ أخذِ العيناتِ**: 250 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

نُهيّئُ بياناتِ MOABB لِتعلّمِ الآلةِ بِالتقطيعِ وإعادةِ التشكيلِ وتوحيدِ القياس.

## المُخرجاتُ المُتوقّعةُ

- مخططانِ يُظهرانِ توزيعَ القيمِ قبلَ وبعدَ التوحيد
- التوزيعُ المُوحّدُ يَتركّزُ حولَ الصفرِ بِعرضٍ مُنتظم

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ |
| --- | --- |
| القنواتُ | 22 |
| العيّناتُ | 1001 |
| الفئاتُ | 2 (left, right) |

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn


<div dir="rtl" align="right">

## 2. تحميلُ مجموعةِ بياناتِ MOABB

تُنزّلُ MOABB البياناتِ تلقائيّاً عندَ أوّلِ استدعاءٍ (حوالي 44 ميجابايت).

</div>

In [ ]:
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
import numpy as np

dataset = BNCI2014_001()
paradigm = MotorImagery(n_classes=2)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

mask = (labels == 'left_hand') | (labels == 'right_hand')
X = X[mask]
labels = labels[mask]

print(f'X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')
print(f'Trials: {len(labels)}')


<div dir="rtl" align="right">

## 3. استكشافُ البياناتِ

</div>

In [ ]:
n_trials, n_channels, n_samples = X.shape
print(f'Trials: {n_trials}')
print(f'Channels: {n_channels}')
print(f'Samples per trial: {n_samples}')
print(f'Trial duration: {n_samples/250:.2f} s')


<div dir="rtl" align="right">

## 4. تطبيقُ التهيئةِ

</div>

In [ ]:
from sklearn.preprocessing import StandardScaler

X_2d = X.reshape(n_trials, n_channels * n_samples)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_2d)
print(f'Reshaped: {X_2d.shape}')
print(f'Scaled mean: {X_scaled.mean():.4f}')
print(f'Scaled std: {X_scaled.std():.4f}')


<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- التوزيعُ الخامُّ مُمتدٌّ بِسعاتٍ مُتفاوتة
- بعدَ التوحيدِ، التوزيعُ مُتمركزٌ حولَ الصفر

</div>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=1, subplot_titles=('Raw distribution', 'Standardized distribution'))
fig.add_trace(go.Histogram(x=X_2d[:, :1000].flatten(), nbinsx=100, name='Raw', marker_color='steelblue'), row=1, col=1)
fig.add_trace(go.Histogram(x=X_scaled[:, :1000].flatten(), nbinsx=100, name='Scaled', marker_color='orange'), row=2, col=1)
fig.update_layout(height=700, title_text='Dataset Preparation - Standardization', showlegend=False)
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- التهيئةُ تَشملُ التقطيعَ وإعادةَ التشكيلِ وتوحيدَ القياس
- التوحيدُ يُضمنُ مساهمةَ جميعِ السماتِ بِنفسِ الوزن
- يَجبُ تطبيقُ التوحيدِ على بياناتِ التدريبِ فقط لِتجنّبِ تَسرّبِ البيانات

</div>